# Evaluation — CS 5542 Challenge 1

Evaluation of **method quality**, distinct from the 418 AC-derived tests, which verify **spec compliance**.

Every table is written to `notebooks/results/` and every figure to `notebooks/figures/`, so the report
quotes measured numbers rather than restating claims.

| Section | Acceptance criterion |
|---|---|
| 1 · Retrieval comparison | AC-13.1 |
| 2 · End-to-end results per profile | AC-13.2 |
| 3 · Latency | AC-13.3 |
| 4 · Scalability | AC-13.4 |
| 5 · Calibration evidence | AC-13.5 |
| 6 · Component sensitivity | AC-13.6 |

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, plotly.express as px, plotly.io as pio
from pathlib import Path
import sys; sys.path.insert(0, "..")

from src.evaluation import (FIGURES, RESULTS, RELEVANCE_PROXY_NOTE, calibration_evidence,
                            component_sensitivity, latency_profile, pooled_candidates,
                            precision_at_k, scalability, semantic_correlation)
from src.filters import warm_caches
from src.indexing import JobIndex
from src.personal_kb import PersonalKB
from src.pipeline import search
from src.profiles import PRESETS

FIGURES.mkdir(parents=True, exist_ok=True); RESULTS.mkdir(parents=True, exist_ok=True)
warm_caches()
ROOT = Path("..").resolve()
jobs = pd.read_parquet(ROOT / "data/processed/jobs_tech.parquet")
index = JobIndex.load_or_build(jobs, index_dir=ROOT / "data/index")
kbs = {k: PersonalKB.build(p.resume_text, p.career_goals) for k, p in PRESETS.items()}
print(f"corpus {len(jobs):,} jobs · {len(PRESETS)} profiles")

## 1 · Retrieval comparison (AC-13.1)

Run on the **unfiltered** corpus — the honest way to compare retrieval methods, since filters would
otherwise do most of the work.

Judgments are **pooled**: run every configuration, take the union of their top-10, judge each job once.
Pooling cannot under-credit a configuration for surfacing a relevant job nobody thought to pre-label.

In [ ]:
print(RELEVANCE_PROXY_NOTE)
rows = []
for key, p in PRESETS.items():
    pool = pooled_candidates(jobs, index, p, key, k=10)
    pool.to_csv(RESULTS / f"pool_{key}.csv", index=False)
    rows.append({"profile": p.name, "pool_size": len(pool), **precision_at_k(pool)})
retrieval = pd.DataFrame(rows); retrieval.to_csv(RESULTS / "retrieval_comparison.csv", index=False)
retrieval

In [ ]:
long = retrieval.melt(id_vars="profile", value_vars=["bm25", "dense", "hybrid"],
                      var_name="retriever", value_name="precision@10")
fig = px.bar(long, x="profile", y="precision@10", color="retriever", barmode="group",
             range_y=[0, 1.05], title="Precision@10 by retrieval method (pooled judgment)")
fig.write_image(FIGURES / "retrieval_comparison.png", scale=2); fig

**Reading this honestly.** The automatic proxy scores almost everything relevant, so it cannot
separate the methods sharply — that is a limitation of the *proxy*, not evidence that the methods are
equivalent. The one signal that does survive is that **dense retrieval alone underperforms BM25** on
the backend and security profiles, which is the asymmetry hybrid retrieval exists to absorb.

The pools are exported to `notebooks/results/pool_*.csv` with an empty `relevant_manual` column;
filling it in and re-running with `label="relevant_manual"` gives the sharper comparison.

## 2 · End-to-end results per profile (AC-13.2)

In [ ]:
rows = []
for key, p in PRESETS.items():
    res = search(jobs, index, p, kbs[key])
    for rank, r in enumerate(res.results, 1):
        rows.append({"profile": p.name, "rank": rank, "score": r["score"], "tier": r["tier"],
                     "title": r["job"]["title"][:46], "location": r["job"]["location_raw"],
                     "matched": len(r["matched_skills"]), "missing": len(r["missing_skills"])})
top5 = pd.DataFrame(rows); top5.to_csv(RESULTS / "top5_by_profile.csv", index=False)
top5

## 3 · Latency (AC-13.3)\n\nMedian and p95 per stage over 20 runs.

In [ ]:
lat = latency_profile(jobs, index, PRESETS["data_science_student"], kbs["data_science_student"], runs=20)
lat.to_csv(RESULTS / "latency.csv", index=False)
fig = px.bar(lat[lat.stage != "total"], x="stage", y="median_ms",
             error_y=lat[lat.stage != "total"]["p95_ms"] - lat[lat.stage != "total"]["median_ms"],
             title="Per-stage latency (median, whisker to p95)")
fig.write_image(FIGURES / "latency.png", scale=2); display(lat); fig

**Filtering before retrieval costs ~57 ms.** The draft plan called stage ordering *"a performance
tuning decision, not an architectural one"* and filtered *after* retrieving 200 candidates. At the
measured survival rates that ordering leaves ~19, ~12 and ~2 candidates to rank — so it is
architectural, and the performance argument runs the other way too.

## 4 · Scalability (AC-13.4)

In [ ]:
sc = scalability()
sc.to_csv(RESULTS / "scalability.csv", index=False)
fig = px.line(sc, x="rows", y="seconds", markers=True,
              title="Ingestion wall-clock vs corpus size (DuckDB, single machine)")
fig.write_image(FIGURES / "scalability.png", scale=2); display(sc); fig

Throughput **rises** with corpus size (7k → 37.6k rows/sec) because fixed startup cost dominates the
small runs. On the 785,741-row `data_jobs` corpus a `GROUP BY` with a median aggregate returns in
0.02 s — roughly 40 M rows/sec.

That is the evidence behind **D9**: at this scale the data fits single-machine, so Spark's JVM startup
and shuffle overhead would be cost without benefit. The crossover is documented rather than asserted.

## 5 · Calibration evidence (AC-13.5)

In [ ]:
cal = calibration_evidence(index, PRESETS["data_science_student"])
cal.to_csv(RESULTS / "calibration_evidence.csv", index=False)
fig = px.histogram(cal, x="raw_cosine", color="population", barmode="overlay", nbins=60,
                   title="Before — raw cosine similarity")
fig.write_image(FIGURES / "calibration_raw.png", scale=2); fig

In [ ]:
fig = px.histogram(cal, x="calibrated", color="population", barmode="overlay", nbins=60,
                   title="After — calibrated against a fixed background")
fig.write_image(FIGURES / "calibration_after.png", scale=2); fig

Raw cosine clusters in a narrow band, so the semantic components would barely vary between jobs.
Calibration spreads the **retrieved candidates** across the full range while compressing the rest of
the corpus toward zero — which is correct: those jobs are irrelevant.

This is the principled answer to the Stage 2 AI code's `sim * 140.0` rescaling
(`agent-exercise:src/matcher.py:117`, whose comment admits the constant was chosen so that
"strong matches reach 85-95%"). Same goal; constants measured from the corpus instead of picked.

## 6 · Component sensitivity (AC-13.6)\n\nZero each component in turn and see whether the top 5 moves.

In [ ]:
sens = component_sensitivity(jobs, index, PRESETS["data_science_student"], kbs["data_science_student"])
sens.to_csv(RESULTS / "component_sensitivity.csv", index=False)
sens["jobs_replaced"] = 5 - sens["top5_overlap"]
fig = px.bar(sens, x="component", y="jobs_replaced",
             title="Top-5 jobs replaced when a component is zeroed (higher = more influential)")
fig.write_image(FIGURES / "component_sensitivity.png", scale=2); display(sens); fig

In [ ]:
corr = pd.DataFrame([{"profile": PRESETS[k].name,
                      "correlation": round(semantic_correlation(jobs, index, PRESETS[k], kbs[k]), 3)}
                     for k in PRESETS])
corr.to_csv(RESULTS / "semantic_correlation.csv", index=False); corr

**Two findings the weights did not anticipate.**

*Education (4%) earns its place.* Zeroing it replaces 2 of 5 results. The weight was originally
justified by an assumption measurement refuted — that most postings state no requirement — and 58.5%
of the corpus does state one. **Q6 closes: keep the weight, discard the reasoning.**

*Location (8%) and preferred skills (8%) change nothing.* Together, 16% of the weight is inert for
this profile. Both have explanations: only 7% of postings list preferred skills, and the top results
are mostly remote, so location is constant across them. Recorded as a limitation rather than re-tuned
one day before the deadline — re-weighting without time to re-validate would be worse than reporting
the measurement.

*Semantic correlation* answers **Q2**: −0.11, −0.00 and +0.79. Largely independent for two profiles,
correlated for the third. Before AC-9.10 excluded the career-goals chunk from the evidence max they
were *identical* on most results — 30% of the weight was one signal counted twice.